# FullStack AI · Ingénierie de l'IA

**Objectif :** LangChain, LangGraph, RAG, bases vectorielles, observabilité, vLLM et MCP.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## LangChain et LangGraph

En Essentials, tu as construit un agent à la main : une boucle qui appelle le LLM, exécute les outils demandés et recommence. C'est la meilleure façon de comprendre. En entreprise, on utilise souvent un **framework** qui fournit les briques toutes faites : **LangChain** pour assembler des chaînes, **LangGraph** pour orchestrer des agents sous forme de graphe. Ce sont les plus répandus, et ceux du programme Jedha.

## LangChain : des briques qui s'emboîtent

LangChain fournit une interface commune pour des dizaines de fournisseurs de LLM, de bases vectorielles et d'outils. Sa syntaxe la plus visible est le **pipe** `|` (LCEL) : la sortie d'une étape devient l'entrée de la suivante.

```python
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu résumes des tickets de support en une phrase."),
    ("user", "{ticket}"),
])
chaine = prompt | ChatAnthropic(model="claude-sonnet-4-5") | StrOutputParser()
print(chaine.invoke({"ticket": "Je n'arrive plus à synchroniser mes fichiers depuis la mise à jour..."}))
chaine.batch([{"ticket": t} for t in tickets])          # en parallèle
```

Chaque brique est un **Runnable** : elle a `invoke`, `batch` et `stream`. L'opérateur `|` crée un nouveau Runnable qui enchaîne les deux. Tu vas le recoder pour voir qu'il n'y a pas de magie.

## LangGraph : un agent est un graphe

Un agent n'est pas une ligne droite : il boucle, il choisit un outil, il peut demander une validation humaine. LangGraph le décrit comme un **graphe d'états** :

- un **état** partagé (un dictionnaire : messages, résultats intermédiaires) ;
- des **nœuds** : des fonctions qui lisent l'état et renvoient des mises à jour ;
- des **arêtes**, fixes ou **conditionnelles** : une fonction regarde l'état et choisit le nœud suivant ;
- un nœud spécial **END** pour s'arrêter.

*Animation interactive « graph » : à voir sur la plateforme web.*

```python
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

def agent(etat: MessagesState):
    return {"messages": [llm_avec_outils.invoke(etat["messages"])]}

def router(etat: MessagesState):
    return "outils" if etat["messages"][-1].tool_calls else END

g = StateGraph(MessagesState)
g.add_node("agent", agent)
g.add_node("outils", ToolNode([meteo, recherche_docs]))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", router)
g.add_edge("outils", "agent")
app = g.compile(checkpointer=memoire)          # mémoire de conversation persistante
app.invoke({"messages": [("user", "Quel temps à Lyon ?")]}, config={"configurable": {"thread_id": "42"}})
```

LangGraph apporte en plus la **persistance** (reprendre une conversation, revenir à un état passé), l'**interruption pour validation humaine** avant une action sensible, et le **streaming** des étapes.

## Framework ou code maison ?

Les frameworks font gagner du temps pour connecter des dizaines de services, mais ils ajoutent des couches d'abstraction qui compliquent le débogage, et ils évoluent vite. Beaucoup d'équipes écrivent leurs agents simples à la main avec le SDK du fournisseur, et réservent LangGraph aux flux complexes avec état. L'important est de savoir ce qui se passe dessous : c'est l'objet des exercices.

> **À retenir** : LangChain = des Runnables composables avec `|`. LangGraph = état + nœuds + arêtes conditionnelles + END. Un agent est une boucle agent → outils → agent jusqu'à ce que le routeur dise stop.

**À faire :**

- [ ] Installer langchain, langchain-anthropic (ou langchain-openai) et langgraph dans un environnement virtuel
- [ ] Écrire une chaîne prompt | llm | parser qui classe des tickets de support, et la lancer en batch sur 20 tickets
- [ ] Reproduire l'agent météo de la leçon avec LangGraph et un vrai outil (API Open-Meteo)
- [ ] Ajouter un checkpointer et une interruption avant l'appel d'un outil sensible, pour valider à la main

### Exercice 1 · Recoder le pipe de LangChain

Écris une classe `Runnable(fonction)` avec : `invoke(x)` qui applique la fonction ; `batch(liste)` qui renvoie la liste des `invoke` ; et l'opérateur `|` (méthode `__or__`) qui renvoie un **nouveau** Runnable appliquant d'abord `self`, puis l'autre. Si l'autre côté du `|` est une simple fonction, enveloppe-la d'abord dans un Runnable.

<details><summary>Indice</summary>

Dans __or__, renvoie Runnable(lambda x: autre.invoke(self.invoke(x))).
</details>

In [ ]:
class Runnable:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
a = Runnable(lambda x: x + 1)
b = Runnable(lambda x: x * 10)
assert a.invoke(1) == 2, "invoke doit appliquer la fonction."
c = a | b
assert isinstance(c, Runnable) and c is not a and c is not b, "a | b doit renvoyer un nouveau Runnable."
assert c.invoke(1) == 20, "a | b applique d'abord a, puis b : (1 + 1) × 10 = 20."
assert (b | a).invoke(1) == 11, "L'ordre compte : b | a donne 1 × 10 + 1 = 11."
assert (a | b | str).invoke(0) == "10", "Une simple fonction à droite du | doit être acceptée."
assert c.batch([0, 1, 2]) == [10, 20, 30], "batch applique invoke à chaque élément."
print("Bravo, tout est correct.")

### Exercice 2 · Un mini LangGraph

Écris une classe `Graphe` avec : `ajouter_noeud(nom, fonction)` (la fonction reçoit l'état et renvoie un dictionnaire de mises à jour) ; `ajouter_arete(source, cible)` ; `ajouter_arete_conditionnelle(source, routeur)` (le routeur reçoit l'état et renvoie le nom du nœud suivant ou `FIN`) ; `point_entree(nom)` ; et `invoke(etat, max_etapes=25)`. `invoke` part d'une **copie** de l'état, exécute les nœuds un par un en fusionnant leurs mises à jour, note chaque nœud visité dans `self.trace` (remise à zéro à chaque appel), s'arrête sur `FIN` (aussi quand un nœud n'a pas de suite) et lève une `RuntimeError` au-delà de `max_etapes` nœuds.

<details><summary>Indice</summary>

Une boucle while courant != FIN : vérifie max_etapes, ajoute à la trace, fusionne les mises à jour (etat.update), puis choisis le suivant (routeur s'il existe, sinon arête fixe, sinon FIN).
</details>

In [ ]:
FIN = "__fin__"

class Graphe:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
g = Graphe()
g.ajouter_noeud("a", lambda e: {"x": e["x"] + 1})
g.ajouter_noeud("b", lambda e: {"y": e["x"] * 2})
g.point_entree("a"); g.ajouter_arete("a", "b")
depart = {"x": 1}
r = g.invoke(depart)
assert r == {"x": 2, "y": 4}, f"a puis b doit donner {{'x': 2, 'y': 4}} ; tu as {r}."
assert depart == {"x": 1}, "invoke ne doit pas modifier l'état passé en argument : travaille sur une copie."
assert g.trace == ["a", "b"], "trace doit lister les nœuds visités, dans l'ordre."
h = Graphe()
h.ajouter_noeud("inc", lambda e: {"n": e["n"] + 1})
h.point_entree("inc")
h.ajouter_arete_conditionnelle("inc", lambda e: FIN if e["n"] >= 3 else "inc")
assert h.invoke({"n": 0})["n"] == 3 and h.trace == ["inc"] * 3, "L'arête conditionnelle doit permettre de boucler jusqu'à FIN."
h.invoke({"n": 2}); assert h.trace == ["inc"], "trace doit être remise à zéro à chaque invoke."
boucle = Graphe(); boucle.ajouter_noeud("x", lambda e: {}); boucle.point_entree("x"); boucle.ajouter_arete("x", "x")
try:
    boucle.invoke({}, max_etapes=5); stoppe = False
except RuntimeError:
    stoppe = True
assert stoppe and len(boucle.trace) == 5, "Au-delà de max_etapes nœuds, lève une RuntimeError."
print("Bravo, tout est correct.")

### Exercice 3 · Un agent avec le graphe

Avec le `Graphe` fourni, construis `agent` : un nœud `"agent"` qui appelle `decider(etat["messages"])` (un LLM simulé fourni) et ajoute sa décision à la liste `messages` ; un nœud `"outils"` qui exécute l'outil demandé par la dernière décision (`OUTILS[nom](**args)`) et ajoute `{"role": "outil", "contenu": resultat}` aux messages ; un routeur après `"agent"` qui va vers `"outils"` si la décision contient la clé `"outil"`, sinon vers `FIN` ; et une arête de `"outils"` vers `"agent"`. Les nœuds renvoient la **nouvelle** liste complète des messages.

<details><summary>Indice</summary>

noeud_agent renvoie {"messages": etat["messages"] + [decider(...)]}. Le routeur regarde "outil" in etat["messages"][-1].
</details>

In [ ]:
FIN = "__fin__"

class Graphe:
    def __init__(self):
        self.noeuds, self.aretes, self.routeurs, self.entree, self.trace = {}, {}, {}, None, []
    def ajouter_noeud(self, nom, fonction):
        self.noeuds[nom] = fonction
    def ajouter_arete(self, source, cible):
        self.aretes[source] = cible
    def ajouter_arete_conditionnelle(self, source, routeur):
        self.routeurs[source] = routeur
    def point_entree(self, nom):
        self.entree = nom
    def invoke(self, etat, max_etapes=25):
        etat, courant, self.trace = dict(etat), self.entree, []
        while courant != FIN:
            if len(self.trace) >= max_etapes:
                raise RuntimeError(f"Plus de {max_etapes} étapes : le graphe tourne en rond.")
            self.trace.append(courant)
            etat.update(self.noeuds[courant](etat) or {})
            courant = self.routeurs[courant](etat) if courant in self.routeurs else self.aretes.get(courant, FIN)
        return etat

def meteo(ville):
    return {"Lyon": "14 °C, nuageux", "Nice": "19 °C, soleil"}.get(ville, "inconnue")

OUTILS = {"meteo": meteo}

def decider(messages):
    """Un LLM simulé : demande la météo, puis répond avec le résultat de l'outil."""
    if messages[-1]["role"] == "outil":
        return {"role": "assistant", "contenu": f"Il fait {messages[-1]['contenu']}."}
    return {"role": "assistant", "outil": "meteo", "args": {"ville": "Lyon"}}

agent = Graphe()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
depart = {"messages": [{"role": "user", "contenu": "Météo à Lyon ?"}]}
fin = agent.invoke(depart)
assert agent.trace == ["agent", "outils", "agent"], f"Parcours attendu : agent → outils → agent ; tu as {agent.trace}."
assert fin["messages"][-1] == {"role": "assistant", "contenu": "Il fait 14 °C, nuageux."}, f"La dernière réponse doit utiliser le résultat de l'outil ; tu as {fin['messages'][-1]}."
assert fin["messages"][2] == {"role": "outil", "contenu": "14 °C, nuageux"}, "Le nœud outils doit ajouter {'role': 'outil', 'contenu': résultat}."
assert len(depart["messages"]) == 1, "Les nœuds doivent renvoyer une nouvelle liste (etat['messages'] + [...]) sans modifier l'ancienne."
print("Bravo, tout est correct.")

---
## Observabilité des LLM

Une application de LLM en production est une boîte noire si on ne l'instrumente pas. Un utilisateur se plaint d'une mauvaise réponse : quel prompt a été envoyé ? Quels documents le RAG a-t-il récupérés ? Combien de temps a pris chaque étape ? Combien a coûté la requête ? L'**observabilité** répond à ces questions.

## Les trois piliers

- **Les traces** : le parcours complet d'une requête, découpé en **spans** (des étapes chronométrées et imbriquées). Pour un RAG : la recherche vectorielle, le reclassement, l'appel au LLM, la vérification de la sortie.
- **Les métriques** : des agrégats dans le temps. Latence (médiane **p50** et **p95**, car la moyenne cache les requêtes très lentes), tokens consommés, coût, taux d'erreur, et des scores de qualité.
- **Les journaux** : les événements détaillés, dont les prompts et réponses complets, en respectant la confidentialité (masquer les données personnelles).

*Animation interactive « trace » : à voir sur la plateforme web.*

## Ce qui est propre aux LLM

| À suivre | Pourquoi |
| --- | --- |
| Prompt et réponse complets, version du prompt | Rejouer et comprendre une mauvaise réponse |
| Tokens d'entrée et de sortie | Le coût, et les limites de contexte |
| Temps jusqu'au premier token (TTFT) | La latence ressentie en streaming |
| Documents récupérés et leur score | Déboguer le RAG : la réponse était-elle dans le contexte ? |
| Appels d'outils et leurs résultats | Déboguer un agent |
| Retours utilisateurs (pouce haut ou bas) | La qualité réelle, au fil du temps |
| Scores d'évaluation en ligne | Un LLM juge note une partie des réponses de production (vu en Essentials) |

## Les outils

- **Langfuse** (open source, auto-hébergeable) et **LangSmith** (de l'équipe LangChain) : traces, coûts, jeux de données d'évaluation, gestion des prompts.
- **OpenTelemetry** : le standard ouvert des traces, avec des conventions dédiées à l'IA générative ; il alimente Datadog, Grafana ou Honeycomb.
- **Arize Phoenix**, **Weights & Biases Weave** : alternatives orientées évaluation.

```python
from langfuse import observe

@observe()                       # chaque appel devient un span, imbriqué automatiquement
def rechercher(question): ...

@observe()
def repondre(question):
    docs = rechercher(question)
    return llm(question, docs)
```

Tu vas recoder ce décorateur, puis analyser des journaux de production.

> **À retenir** : traces (spans imbriqués), métriques (p50, p95, tokens, coût, erreurs) et journaux. Pour un LLM : garder prompts, réponses, documents et retours, et suivre la qualité en continu.

**À faire :**

- [ ] Créer un compte Langfuse (ou le lancer avec Docker) et tracer ton RAG d'Essentials avec @observe
- [ ] Construire dans Langfuse un tableau de bord coût et latence p95 par jour
- [ ] Ajouter un pouce haut ou bas dans ton interface et l'envoyer comme score à la trace

### Exercice 1 · Un décorateur de traçage

Écris le décorateur `tracer` (sans argument). Chaque appel de la fonction décorée ajoute à la liste globale `SPANS` un dictionnaire `{"nom": nom_de_la_fonction, "parent": nom_du_span_englobant_ou_None, "duree_ms": float, "statut": "ok" ou "erreur"}`. Utilise une pile (`PILE`, une liste) pour connaître le span englobant. En cas d'exception, note le statut `"erreur"` puis laisse l'exception remonter. Le span est ajouté à `SPANS` quand la fonction **se termine**.

<details><summary>Indice</summary>

Le parent est PILE[-1] avant d'empiler le nom courant. Un try / except / finally : le finally dépile et enregistre le span.
</details>

In [ ]:
import time
import functools

SPANS = []
PILE = []

def tracer(f):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import time
SPANS.clear(); PILE.clear()
@tracer
def feuille():
    time.sleep(0.01); return 1
@tracer
def racine():
    return feuille() + feuille()
assert racine() == 2, "La fonction décorée doit renvoyer le résultat de la fonction d'origine."
assert [s["nom"] for s in SPANS] == ["feuille", "feuille", "racine"], f"Les spans sont ajoutés à la fin de chaque appel : feuille, feuille, racine ; tu as {[s['nom'] for s in SPANS]}."
assert [s["parent"] for s in SPANS] == ["racine", "racine", None], "Le parent d'un span est le nom de la fonction tracée qui l'englobe (None pour la racine)."
assert SPANS[2]["duree_ms"] >= 20 and SPANS[0]["duree_ms"] >= 10, "duree_ms doit être en millisecondes."
assert all(s["statut"] == "ok" for s in SPANS), "Statut ok quand tout se passe bien."
@tracer
def casse():
    raise ValueError("panne")
try:
    casse(); remonte = False
except ValueError:
    remonte = True
assert remonte and SPANS[-1]["statut"] == "erreur", "En cas d'exception : statut erreur, et l'exception remonte."
assert PILE == [], "La pile doit être vidée même après une erreur (utilise finally)."
assert racine.__name__ == "racine", "Utilise functools.wraps pour garder le nom de la fonction."
print("Bravo, tout est correct.")

### Exercice 2 · Analyser les journaux de production

Le DataFrame `journaux` (une ligne par appel de LLM) et le dictionnaire `PRIX` (euros par million de tokens d'entrée et de sortie) sont fournis. Calcule `resume`, un DataFrame indexé par `modele` avec les colonnes `appels`, `p50_ms` et `p95_ms` (quantiles 0,5 et 0,95 de `latence_ms`), `taux_erreur` (moyenne de `erreur`) et `cout_eur` (somme des coûts des appels). Puis `alertes`, la liste triée des modèles dont le p95 dépasse 4 000 ms **ou** le taux d'erreur dépasse 2 %.

<details><summary>Indice</summary>

Ajoute d'abord une colonne de coût par appel, puis groupby("modele") et un DataFrame de séries : g.size(), g["latence_ms"].quantile(0.95), etc.
</details>

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 3000
modele = rng.choice(["rapide", "standard", "expert"], n, p=[0.5, 0.35, 0.15])
base = pd.Series(modele).map({"rapide": 700, "standard": 1600, "expert": 3200}).to_numpy()
journaux = pd.DataFrame({
    "modele": modele,
    "latence_ms": np.round(base * rng.lognormal(0, 0.35, n)),
    "tokens_entree": rng.integers(300, 4000, n),
    "tokens_sortie": rng.integers(50, 800, n),
    "erreur": rng.random(n) < pd.Series(modele).map({"rapide": 0.005, "standard": 0.03, "expert": 0.01}).to_numpy(),
})
PRIX = {"rapide": (0.25, 1.25), "standard": (3.0, 15.0), "expert": (15.0, 75.0)}   # € par million (entrée, sortie)

resume = None
alertes = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert resume is not None and set(["appels", "p50_ms", "p95_ms", "taux_erreur", "cout_eur"]) <= set(resume.columns), "resume doit avoir les colonnes appels, p50_ms, p95_ms, taux_erreur, cout_eur."
assert sorted(resume.index) == ["expert", "rapide", "standard"], "resume doit être indexé par modele (groupby)."
for m in ["rapide", "standard", "expert"]:
    s = journaux[journaux["modele"] == m]
    assert resume.loc[m, "appels"] == len(s), f"Nombre d'appels faux pour {m}."
    assert abs(resume.loc[m, "p95_ms"] - s["latence_ms"].quantile(0.95)) < 1e-6, f"p95_ms faux pour {m} : quantile(0.95)."
    assert abs(resume.loc[m, "p50_ms"] - s["latence_ms"].median()) < 1e-6, f"p50_ms faux pour {m}."
    c = ((s["tokens_entree"] * PRIX[m][0] + s["tokens_sortie"] * PRIX[m][1]) / 1e6).sum()
    assert abs(resume.loc[m, "cout_eur"] - c) < 1e-6, f"cout_eur faux pour {m} : (tokens_entree × prix_entrée + tokens_sortie × prix_sortie) / 1 000 000."
    assert abs(resume.loc[m, "taux_erreur"] - s["erreur"].mean()) < 1e-9, f"taux_erreur faux pour {m}."
att = sorted(m for m in resume.index if resume.loc[m, "p95_ms"] > 4000 or resume.loc[m, "taux_erreur"] > 0.02)
assert alertes == att, f"alertes attendues : {att} ; tu as {alertes}."
print("Bravo, tout est correct.")

---
## Servir un LLM : vLLM et Hugging Face

Appeler l'API d'Anthropic ou d'OpenAI est le plus simple. Mais certaines entreprises doivent faire tourner un modèle **sur leurs propres serveurs** : données qui ne doivent pas sortir, coût à grande échelle, modèle affiné maison. Il faut alors un **serveur d'inférence**, et comprendre ce qui limite ses performances.

## Deux phases, deux goulots

Générer une réponse se fait en deux temps :

1. **Le préremplissage** (*prefill*) : tout le prompt est traité d'un coup, en parallèle. C'est limité par la **puissance de calcul**. Il détermine le **temps jusqu'au premier token** (TTFT).
2. **Le décodage** : un token à la fois, chacun dépendant du précédent. À chaque pas, il faut relire tous les poids du modèle en mémoire : c'est limité par la **bande passante mémoire**, pas par le calcul. Il détermine la vitesse en tokens par seconde.

Conséquence : traiter un seul utilisateur à la fois gaspille le GPU. Le secret, c'est de **regrouper** les requêtes (*batching*) : relire les poids une fois pour avancer de 32 conversations au lieu d'une.

## Le cache KV : le vrai facteur limitant

Tu as codé le cache KV dans le projet mini-GPT. Sa taille pour une séquence :

$$
\text{mémoire KV} = 2 \times n_{\text{couches}} \times n_{\text{têtes KV}} \times d_{\text{tête}} \times n_{\text{tokens}} \times \text{octets}
$$

Le facteur 2 compte les clés et les valeurs. Pour Llama 3 8B en 16 bits, c'est 128 Ko par token : une conversation de 8 000 tokens occupe 1 Go. Les poids prennent 16 Go ; ce qui reste sur le GPU fixe le nombre de conversations simultanées. D'où les astuces des modèles récents : **GQA** (moins de têtes pour K et V que pour Q), la quantification du cache, les fenêtres glissantes.

## Ce que fait vLLM

**vLLM** (Berkeley, 2023) est le serveur open source le plus utilisé. Deux idées clés :

- **PagedAttention** : le cache KV est découpé en petits blocs, comme la mémoire virtuelle d'un système d'exploitation. Plus de gaspillage à réserver la longueur maximale pour chaque requête.
- **Le batching continu** : dès qu'une conversation se termine, une nouvelle prend sa place dans le lot, sans attendre que tout le lot ait fini.

*Animation interactive « batching » : à voir sur la plateforme web.*

```python
pip install vllm
vllm serve meta-llama/Llama-3.1-8B-Instruct --max-model-len 8192 --gpu-memory-utilization 0.9

# vLLM expose une API compatible OpenAI : le même client fonctionne
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="inutile")
r = client.chat.completions.create(model="meta-llama/Llama-3.1-8B-Instruct", messages=[{"role": "user", "content": "Bonjour"}])
```

## Les autres options

| Outil | Pour quoi |
| --- | --- |
| **Ollama**, **llama.cpp** | Faire tourner un modèle quantifié sur un portable, même sans GPU |
| **TGI** (Hugging Face), **SGLang**, **TensorRT-LLM** (NVIDIA) | Serveurs de production, alternatives à vLLM |
| **Inference Endpoints** de Hugging Face, **Bedrock**, **SageMaker** | Un modèle ouvert déployé et géré pour toi |

> **À retenir** : préremplissage limité par le calcul, décodage limité par la mémoire. Le cache KV fixe le nombre d'utilisateurs simultanés. vLLM : PagedAttention + batching continu + API compatible OpenAI.

**À faire :**

- [ ] Installer Ollama, télécharger un petit modèle (llama3.2 ou qwen2.5) et l'interroger en local avec son API
- [ ] Sur Google Colab avec un GPU, lancer vLLM avec un modèle de 1 à 3 milliards de paramètres et mesurer les tokens par seconde avec 1 puis 16 requêtes simultanées
- [ ] Appeler ton serveur vLLM avec le client openai en changeant seulement base_url

### Exercice 1 · Dimensionner le cache KV

Écris `memoire_kv(n_couches, n_tetes_kv, d_tete, n_tokens, octets=2)`, la taille du cache KV en octets. Puis, pour Llama 3 8B (32 couches, 8 têtes KV, têtes de dimension 128, en 16 bits), calcule `par_token_ko` (la taille par token, en Kio : divise par 1024) et `sequences_max`, le nombre entier de conversations de 8 192 tokens qui tiennent sur un GPU de 80 Gio utilisé à 90 %, une fois retirés les 16 Gio des poids (1 Gio = 1024³ octets).

<details><summary>Indice</summary>

Le cache d'une conversation fait memoire_kv(32, 8, 128, 8192) octets ; la mémoire libre fait (80 × 0.9 − 16) × 1024³ octets.
</details>

In [ ]:
def memoire_kv(n_couches, n_tetes_kv, d_tete, n_tokens, octets=2):
    pass

par_token_ko = None
sequences_max = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert memoire_kv(1, 1, 1, 1, 1) == 2, "Le facteur 2 compte les clés et les valeurs."
assert memoire_kv(32, 8, 128, 1) == 131072, "Pour Llama 3 8B en 16 bits : 2 × 32 × 8 × 128 × 2 = 131 072 octets par token."
assert par_token_ko == 128, f"par_token_ko doit valoir 128 ; tu as {par_token_ko}."
attendu = int((80 * 0.9 - 16) * 1024 ** 3 // (131072 * 8192))
assert sequences_max == attendu, f"sequences_max doit valoir {attendu} : (80 × 0,9 − 16) Gio divisés par la taille du cache d'une conversation de 8 192 tokens."
print("Bravo, tout est correct.")

### Exercice 2 · Batching statique contre continu

Chaque requête doit générer `longueurs[i]` tokens ; le serveur avance de 1 token par requête active à chaque pas, avec au plus `B` requêtes actives. Écris `pas_statique(longueurs, B)` : les requêtes sont traitées par lots consécutifs de `B`, et un lot dure autant que sa plus longue requête. Puis `pas_continu(longueurs, B)` : dès qu'une requête se termine, la suivante dans la file prend sa place au pas suivant. Les deux renvoient le nombre total de pas.

<details><summary>Indice</summary>

Statique : somme des maximums de chaque tranche longueurs[i:i + B]. Continu : simule avec une file et une liste de tokens restants par requête active.
</details>

In [ ]:
def pas_statique(longueurs, B):
    pass

def pas_continu(longueurs, B):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import random
def _ref(L, B):
    file, act, p = list(L), [], 0
    while file or act:
        while file and len(act) < B: act.append(file.pop(0))
        p += 1; act = [r - 1 for r in act if r > 1]
    return p
assert pas_statique([3, 5, 2, 2], 2) == 7, "Lots [3, 5] puis [2, 2] : 5 + 2 = 7 pas."
assert pas_statique([5, 1, 1, 1, 1], 2) == 7, "Lots [5, 1], [1, 1], [1] : 5 + 1 + 1 = 7 pas."
assert pas_continu([5, 1, 1, 1, 1], 2) == 5, "En continu, [5, 1, 1, 1, 1] avec B = 2 prend 5 pas : pendant que la longue requête tourne, les courtes se succèdent dans la deuxième place."
rng = random.Random(1)
for _ in range(20):
    L = [rng.randint(1, 60) for _ in range(rng.randint(1, 15))]; B = rng.randint(1, 5)
    assert pas_continu(L, B) == _ref(L, B), f"pas_continu({L}, {B}) devrait valoir {_ref(L, B)}."
    assert pas_continu(L, B) <= pas_statique(L, B), "Le batching continu ne fait jamais pire que le statique."
assert pas_statique([4], 8) == 4 and pas_continu([4], 8) == 4, "Une seule requête de 4 tokens prend 4 pas."
print("Bravo, tout est correct.")

---
## MCP : le Model Context Protocol

Chaque application d'IA veut se connecter à des outils : Google Drive, GitHub, une base de données, le CRM de l'entreprise. Sans standard, chaque couple application × outil demande son propre connecteur. Le **Model Context Protocol** (MCP), publié par Anthropic fin 2024 et adopté depuis par OpenAI, Google, Microsoft et la plupart des éditeurs, règle ce problème : un outil écrit **une fois** comme serveur MCP fonctionne dans **toutes** les applications compatibles. On le compare souvent à un port USB-C pour l'IA.

## Les trois rôles

- **L'hôte** : l'application que voit l'utilisateur (Claude Desktop, un IDE, ton propre agent).
- **Le client** : dans l'hôte, il gère la connexion à un serveur.
- **Le serveur** : un petit programme qui expose des capacités. Il peut tourner en local (communication par **stdio**) ou à distance (par **HTTP**).

## Ce qu'un serveur expose

| Primitive | Qui décide de l'utiliser | Exemple |
| --- | --- | --- |
| **Outils** (*tools*) | Le modèle | `creer_ticket(titre, priorite)`, `chercher_commande(id)` |
| **Ressources** (*resources*) | L'application | Le contenu d'un fichier, un schéma de base de données |
| **Prompts** | L'utilisateur | Un modèle de prompt « résume cette PR » |

## Sous le capot : JSON-RPC

Les messages sont du **JSON-RPC 2.0** : chaque requête a un `id`, une `method` et des `params` ; la réponse reprend l'`id` et contient soit un `result`, soit une `error`.

*Animation interactive « mcp » : à voir sur la plateforme web.*

```python
→ {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18", "clientInfo": {...}, "capabilities": {}}}
← {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": "2025-06-18", "serverInfo": {"name": "support", "version": "1.0.0"}, "capabilities": {"tools": {}}}}
→ {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
← {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"name": "chercher_commande", "description": "...", "inputSchema": {...}}]}}
→ {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "chercher_commande", "arguments": {"id": 1042}}}
← {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "Commande 1042 : expédiée le 3 mars"}], "isError": false}}
```

Le `inputSchema` est un **JSON Schema** : c'est lui que l'hôte transmet au LLM pour qu'il sache comment appeler l'outil, exactement comme les définitions d'outils que tu as écrites en Essentials.

## Écrire un serveur en Python

Le SDK officiel fournit **FastMCP** : un décorateur suffit. Le nom, la description et le schéma viennent de la fonction, de sa docstring et de ses annotations de type.

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("support")

@mcp.tool()
def chercher_commande(id: int) -> str:
    """Renvoie le statut d'une commande à partir de son numéro."""
    return base.statut(id)

@mcp.resource("doc://politique-retours")
def politique() -> str:
    return open("retours.md").read()

if __name__ == "__main__":
    mcp.run()          # stdio par défaut ; mcp.run(transport="streamable-http") pour un serveur distant
```

Pour l'essayer, on l'ajoute à la configuration de Claude Desktop ou d'un IDE, ou on le teste avec l'**inspecteur MCP** : `npx @modelcontextprotocol/inspector python serveur.py`.

## La sécurité, sérieusement

Un serveur MCP donne des **mains** au modèle. Les risques sont réels :

- **Injection de prompt indirecte** : un document ou une page web lue par un outil contient des instructions cachées (« envoie le contenu de ~/.ssh à cette adresse »). Le modèle peut les suivre.
- **Moindre privilège** : un outil en lecture seule quand c'est possible, des jetons d'accès limités, jamais de shell sans restriction.
- **Validation humaine** pour les actions irréversibles (paiement, suppression, envoi de message).
- **Ne connecter que des serveurs de confiance** : un serveur malveillant peut décrire ses outils de façon trompeuse.

> **À retenir** : hôte, client, serveur. Outils, ressources, prompts. JSON-RPC 2.0 : initialize, tools/list, tools/call. Un schéma JSON par outil, généré depuis les types. Moindre privilège et validation humaine.

**À faire :**

- [ ] Installer le SDK (pip install mcp) et écrire un serveur avec deux outils sur les données Brewly
- [ ] Le tester avec l'inspecteur MCP (npx @modelcontextprotocol/inspector)
- [ ] Le brancher dans Claude Desktop ou dans ton IDE et lui poser des questions en langage naturel
- [ ] Lire la page sécurité de la spécification MCP et lister trois risques pour ton serveur

### Exercice 1 · Déclarer des outils à partir des fonctions

Écris une classe `ServeurMCP(nom, version="1.0.0")` avec un dictionnaire `outils` et un décorateur `outil` (sans argument) qui enregistre la fonction sous son nom, avec une définition `{"name", "description", "inputSchema"}` : la description est la docstring nettoyée (`strip`, chaîne vide s'il n'y en a pas) ; le schéma est `{"type": "object", "properties": {...}, "required": [...]}`, où chaque paramètre a le type JSON correspondant à son annotation (`int` → `integer`, `float` → `number`, `str` → `string`, `bool` → `boolean`, `string` par défaut) et où `required` liste les paramètres sans valeur par défaut. Le décorateur renvoie la fonction inchangée. Ajoute `lister()` qui renvoie la liste des définitions.

<details><summary>Indice</summary>

inspect.signature(f).parameters donne chaque paramètre avec .annotation et .default ; inspect.Parameter.empty signale l'absence de valeur par défaut.
</details>

In [ ]:
import inspect

class ServeurMCP:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
s = ServeurMCP("test")
@s.outil
def rembourser(commande: int, montant: float, motif: str = "", urgent: bool = False):
    """
    Rembourse une commande.
    """
    return "ok"
assert rembourser(1, 2.0) == "ok", "Le décorateur doit renvoyer la fonction inchangée."
assert "rembourser" in s.outils, "L'outil doit être enregistré sous le nom de la fonction."
d = s.lister()
assert isinstance(d, list) and len(d) == 1, "lister() renvoie la liste des définitions."
d = d[0]
assert d["name"] == "rembourser" and d["description"] == "Rembourse une commande.", "name = nom de la fonction ; description = docstring nettoyée avec strip()."
sch = d["inputSchema"]
assert sch["type"] == "object" and sch["properties"] == {"commande": {"type": "integer"}, "montant": {"type": "number"}, "motif": {"type": "string"}, "urgent": {"type": "boolean"}}, f"Propriétés inattendues : {sch['properties']}."
assert sch["required"] == ["commande", "montant"], "required ne liste que les paramètres sans valeur par défaut."
@s.outil
def sans_doc(x):
    return x
assert s.lister()[1]["description"] == "", "Sans docstring, la description est une chaîne vide."
assert s.lister()[1]["inputSchema"]["properties"] == {"x": {"type": "string"}}, "Sans annotation, le type par défaut est string."
print("Bravo, tout est correct.")

### Exercice 2 · Répondre au protocole

La classe `ServeurMCP` de l'exercice précédent est fournie. Ajoute-lui (dans une sous-classe `Serveur`) la méthode `traiter(requete)` qui reçoit un message JSON-RPC (un dictionnaire) et renvoie la réponse. Toute réponse contient `"jsonrpc": "2.0"` et l'`id` de la requête. `initialize` → `result` avec `protocolVersion` (`"2025-06-18"`), `serverInfo` (`name` et `version`) et `capabilities` (`{"tools": {}}`). `tools/list` → `result` = `{"tools": self.lister()}`. `tools/call` → appelle l'outil avec `params["arguments"]` et renvoie `result` = `{"content": [{"type": "text", "text": str(resultat)}], "isError": False}` ; si l'outil lève une exception ou n'existe pas, même forme avec le message d'erreur en texte et `isError: True`. Méthode inconnue → `error` = `{"code": -32601, "message": "Method not found"}`. Une notification (message sans `id`) ne reçoit pas de réponse : renvoie `None`.

<details><summary>Indice</summary>

Commence par traiter l'absence d'id. Prépare base = {"jsonrpc": "2.0", "id": ...} et complète-la selon la méthode avec {**base, "result": ...}.
</details>

In [ ]:
import inspect

TYPES_JSON = {int: "integer", float: "number", str: "string", bool: "boolean", list: "array", dict: "object"}

class ServeurMCP:
    def __init__(self, nom, version="1.0.0"):
        self.nom, self.version, self.outils = nom, version, {}

    def outil(self, f):
        sig = inspect.signature(f)
        proprietes = {n: {"type": TYPES_JSON.get(p.annotation, "string")} for n, p in sig.parameters.items()}
        requis = [n for n, p in sig.parameters.items() if p.default is inspect.Parameter.empty]
        self.outils[f.__name__] = {
            "fonction": f,
            "definition": {"name": f.__name__, "description": (f.__doc__ or "").strip(),
                           "inputSchema": {"type": "object", "properties": proprietes, "required": requis}},
        }
        return f

    def lister(self):
        return [o["definition"] for o in self.outils.values()]

class Serveur(ServeurMCP):
    def traiter(self, requete):
        pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
s = Serveur("support", "2.1.0")
@s.outil
def diviser(a: float, b: float):
    """Divise a par b."""
    return a / b
r = s.traiter({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}})
assert r is not None and r["jsonrpc"] == "2.0" and r["id"] == 1, "Chaque réponse reprend jsonrpc 2.0 et l'id de la requête."
assert r["result"]["serverInfo"] == {"name": "support", "version": "2.1.0"} and r["result"]["capabilities"] == {"tools": {}} and r["result"]["protocolVersion"] == "2025-06-18", "initialize doit renvoyer protocolVersion, serverInfo et capabilities."
r = s.traiter({"jsonrpc": "2.0", "id": "a", "method": "tools/list"})
assert r["id"] == "a" and [t["name"] for t in r["result"]["tools"]] == ["diviser"], "tools/list renvoie {'tools': self.lister()}."
r = s.traiter({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "diviser", "arguments": {"a": 6, "b": 4}}})
assert r["result"] == {"content": [{"type": "text", "text": "1.5"}], "isError": False}, f"tools/call doit renvoyer le résultat en texte ; tu as {r.get('result')}."
r = s.traiter({"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "diviser", "arguments": {"a": 1, "b": 0}}})
assert r["result"]["isError"] is True and "division" in r["result"]["content"][0]["text"], "Une exception de l'outil devient un résultat avec isError: True et le message en texte."
r = s.traiter({"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "inexistant", "arguments": {}}})
assert r["result"]["isError"] is True, "Un outil inconnu donne aussi isError: True."
r = s.traiter({"jsonrpc": "2.0", "id": 6, "method": "resources/teleporter"})
assert r == {"jsonrpc": "2.0", "id": 6, "error": {"code": -32601, "message": "Method not found"}}, "Une méthode inconnue donne une error -32601."
assert s.traiter({"jsonrpc": "2.0", "method": "notifications/initialized"}) is None, "Une notification (sans id) ne reçoit pas de réponse."
print("Bravo, tout est correct.")